# Option Exercise-Policy Evaluation

## 1. Research scope

I study how an exercise policy should be assessed when its decisions, the martingale used to bound its loss, and the computational precision of its assessment can each change. This historical notebook connects the [archived research paper](https://github.com/Amanjeet-S/Option-Exercise-Policy-Evaluation/blob/v1.0.0/paper/Exercise_Policy_Loss_under_Computational_and_Financial_Constraints.pdf), its [LaTeX source](https://github.com/Amanjeet-S/Option-Exercise-Policy-Evaluation/blob/v1.0.0/paper/main.tex), the [released numerical findings](../paper/numerical_results.md), the [CPU timing supplement](../paper/cpu_timings.md) and the [replication entry point](../replication/run.py).

The computations have three evidential roles. The first calculations execute the financial model on stipulated mathematical inputs and enumerate finite probability laws exactly. Those inputs are not observations, historical estimates or substitutes for missing market records. The allocation table checks the presentation and arithmetic of released financial aggregates without repeating their experiments. The CPU calculations reconstruct the forecast comparisons, budget failures and expenditure from eighteen released workload records containing original computational measurements. They do not rerun those workloads.

No raw subscription observations, network requests or private acquisition records are read. Section 7 nevertheless displays historical source-dependent summaries, which retain their LSEG provenance and publication conditions. Independent historical market validation is outside the completed study's evidential scope. Missing historical funding constituents therefore do not prevent execution of this notebook.

In [1]:
from fractions import Fraction as Q
import importlib.util
import json
from pathlib import Path
import sys

import numpy as np
from IPython.display import Markdown, display

ROOT = next(p for p in (Path.cwd(), *Path.cwd().parents)
            if (p / 'src/qf_research').is_dir())
if str(ROOT/'src') not in sys.path:
    sys.path.insert(0, str(ROOT/'src'))
PROJECT = ROOT/'.'


def module_from_file(name, path):
    specification = importlib.util.spec_from_file_location(name, path)
    module = importlib.util.module_from_spec(specification)
    sys.modules[name] = module
    specification.loader.exec_module(module)
    return module


def show_table(headers, rows):
    lines = ['| ' + ' | '.join(headers) + ' |',
             '| ' + ' | '.join('---' for _ in headers) + ' |']
    lines += ['| ' + ' | '.join(str(value) for value in row) + ' |' for row in rows]
    display(Markdown('\n'.join(lines)))

probability = module_from_file('notebook_allocation_probability',
    PROJECT/'paper/verification/allocation_probability.py')
summary_checks = module_from_file('notebook_numerical_results',
    PROJECT/'paper/verification/numerical_results.py')

## 2. An analytically identifiable boundary case

For terminal exercise only, the put payoff is piecewise linear. Its conditional lognormal expectation supplies an analytical reference. Centring its terminal payoff gives a martingale for which the adjusted terminal reward is constant. There is no maximisation over competing stopping dates, so the dual gap is exactly zero in mathematical arithmetic.

I use this limiting case to test the implementation against a known value before assessing a learned policy. The following initial state, volatility and settlement factors are chosen mathematical constants. The tolerance concerns floating-point evaluation, rather than a probability statement about market prices. Analytical centring and optimal-stopping duality follow Haugh and Kogan (2004, Section 3.2).

In [2]:
from qf_research.optimal_stopping.contracts import Option
from qf_research.optimal_stopping.funded_cash import (
    FundedCashModel, FundedExercise, european_reference)
from qf_research.optimal_stopping.funded_policy import simulate_paths, fit_policy
from qf_research.optimal_stopping.value_martingale import (
    build_values, observation_rights, value_martingale, evaluate_value_dual)

terminal_model = FundedCashModel(101., .2, 1., .98)
terminal_option = Option(100., 1., 'put')
terminal_rights = (FundedExercise(1., .97),)
terminal_values = build_values(terminal_model, terminal_option, terminal_rights,
                              core_steps=40, tolerance=.001)
training = simulate_paths(terminal_model, terminal_option, terminal_rights, 50,
                          np.random.default_rng(41))
terminal_policy = fit_policy(terminal_option, training, 2)
evaluation = simulate_paths(terminal_model, terminal_option,
    observation_rights(terminal_values), 2000, np.random.default_rng(42))
terminal_result = evaluate_value_dual(
    terminal_policy, value_martingale(terminal_values, evaluation))
analytical_value = european_reference(
    terminal_model, terminal_option, terminal_rights[-1]).present_value
maximum_error = float(np.max(np.abs(terminal_result.upper_payoffs-analytical_value)))
assert maximum_error < 3e-12
assert terminal_result.gap_bound.value == terminal_result.gap_bound.standard_error == 0.
show_table(['Mathematical validation quantity', 'Computed value'], [
    ('Analytical present value', f'{analytical_value:.10f}'),
    ('Largest adjusted-payoff error', f'{maximum_error:.3g}'),
    ('Expected policy-loss upper bound', f'{terminal_result.gap_bound.value:.1f}')])

| Mathematical validation quantity | Computed value |
| --- | --- |
| Analytical present value | 6.0457695714 |
| Largest adjusted-payoff error | 7.11e-15 |
| Expected policy-loss upper bound | 0.0 |

## 3. A complete funded-model calculation

The discounted ownership state $Y$ follows $\mathrm dY_t=\sigma Y_t\,\mathrm dW_t$ between cash events. A stipulated discounted cash cap $c$ transfers $\min(Y_{t-},c)$ into a receivable and leaves $Y_t=(Y_{t-}-c)^+$. Ownership plus detached receivables is the martingale gain process; the ex-cash ownership state alone is not a martingale across the jump. For a put with strike $K$, exercise settlement factor $f_j$ and premium factor $f_p$, the discounted exercise reward is $(Kf_j-Y_j)^+$ and its quoted premium value is the present value divided by $f_p$.

The replication entry point specifies cash events, exercise dates and settlement factors before simulation. It fits two policy procedures, evaluates them on fresh shared paths and compares their martingale-adjusted rewards. The small configuration is intended to exercise the complete workflow in a reader's environment. Its sample counts are deliberately modest and do not reproduce the published research estimates.

Independent training and evaluation matter because a fitted stopping rule must be fixed before its evaluation paths are observed. On the enlarged filtration containing the training record at time zero, the evaluation innovations remain independent of that record. Integrability and the finite calendar then permit optional sampling of the centred martingale. This identifies the expected adjusted reward with the value of the fitted policy.

Each reported gap is an estimate of an expectation. Pathwise non-negativity of the upper-minus-policy payoff is a separate exact construction property. The configuration is labelled as a stipulated fixture; no financial parameter is inferred from this run.

In [3]:
replication = module_from_file('notebook_conditional_replication', PROJECT/'replication/run.py')
fixture = replication.load_specification(PROJECT/'replication/configs/stipulated_fixture.json')
fixture_results = replication.run(fixture)
# The following cells display selected research quantities, not source metadata.
assert fixture_results['classification'] == 'stipulated_mathematical_fixture'
assert fixture_results['design']['counts_fixed_before_execution'] is True
summary_rows = []
for row in fixture_results['numerics']['training_evaluation']:
    estimate = row['estimate']
    summary_rows.append((row['evaluation_paths'], estimate['observations'],
        f"{estimate['value']:.6f}", f"{estimate['standard_error']:.6f}",
        f"{row['training_variance_moment_estimate']:.6f}"))
show_table(['Paths per pair', 'Training pairs', 'Mean policy gain',
            'Between-pair standard error', 'Signed training component estimate'], summary_rows)
terminal = fixture_results['numerics']['terminal_adjustment']
assert terminal['minimum_integrated_gain'] >= -1e-11
assert terminal['maximum_common_upper_identity_error'] < 1e-10
show_table(['Terminal-adjustment quantity', 'Stipulated-model result'], [
    ('Conditional gain, averaged over evaluation paths', f"{terminal['integrated_gain']['value']:.6f}"),
    ('Realised paired gain, averaged over evaluation paths', f"{terminal['realised_gain']['value']:.6f}"),
    ('Largest common-upper identity error', f"{terminal['maximum_common_upper_identity_error']:.3g}")])

| Paths per pair | Training pairs | Mean policy gain | Between-pair standard error | Signed training component estimate |
| --- | --- | --- | --- | --- |
| 256 | 4 | 0.024216 | 0.012133 | 0.000485 |
| 1024 | 4 | 0.019429 | 0.017979 | 0.001260 |

| Terminal-adjustment quantity | Stipulated-model result |
| --- | --- |
| Conditional gain, averaged over evaluation paths | 0.274321 |
| Realised paired gain, averaged over evaluation paths | 0.246500 |
| Largest common-upper identity error | 4.44e-16 |

The gain and its standard error are expressed in this fixture's abstract premium units. Four training pairs are insufficient for a reliable assessment of interval coverage or a general ranking of training procedures. The signed training component subtracts estimated evaluation noise from the variance across pair means; a negative realisation would be retained rather than interpreted as a negative population variance.

The terminal adjustment preserves earlier instructions and waits until the later endpoint on paths that survive to the original last decision. No new stopping decision is fitted. Its integrated conditional gain is non-negative. The realised paired gain also contains terminal evaluation noise, so equality of the two realised averages is neither expected nor imposed. A common upper payoff makes the reduction in the gap equal to the gain at the comparison level checked by the implementation.

## 4. Repeated training and conditional variance

Let $D$ denote a pair of training samples and let $Z_D$ denote the difference of their martingale-adjusted exercise rewards on one common evaluation path. With $\Delta(D)=\mathbb E[Z_D\mid D]$, write

$$A=\operatorname{Var}\{\Delta(D)\},\qquad B=\mathbb E\{\operatorname{Var}(Z_D\mid D)\}.$$

For $R$ independent training pairs, each evaluated on $N$ conditionally independent paths, iterated expectation and the law of total variance give

$$\mathbb E[\widehat\mu_{R,N}]=\mathbb E[Z_D],\qquad
\operatorname{Var}(\widehat\mu_{R,N})=\frac{A}{R}+\frac{B}{RN}.$$

The calculation separates uncertainty across fitted policies from noise within a policy comparison. Increasing $N$ cannot remove the first component. It does not imply that the two policies within one pair, or their rewards on common paths, are independent.

I check this identity by enumerating the complete probability law with rational arithmetic. The stipulated law permits dependence between the two training samples and a conditional evaluation variance that changes with the training outcome. Exhaustive enumeration checks this instance independently of the variance formula; it is not a proof of the general measure-theoretic result.

In [4]:
training_law, conditional_gains = probability.training_fixture('coupled')
components = probability.variance_components(training_law, conditional_gains)
exact_rows = []
for pairs, paths in ((1, 1), (1, 3), (2, 1), (2, 3)):
    law = probability.repeated_training_mean(training_law, conditional_gains, pairs, paths)
    formula = components.a/pairs + components.beta/(pairs*paths)
    assert law.expectation() == components.mean
    assert law.variance() == formula
    exact_rows.append((pairs, paths, str(law.expectation()), str(law.variance()), str(formula)))
show_table(['Pairs R', 'Paths N', 'Exact mean', 'Enumerated variance', 'Formula'], exact_rows)

| Pairs R | Paths N | Exact mean | Enumerated variance | Formula |
| --- | --- | --- | --- | --- |
| 1 | 1 | 1/3 | 38/9 | 38/9 |
| 1 | 3 | 1/3 | 2 | 2 |
| 2 | 1 | 1/3 | 19/9 | 19/9 |
| 2 | 3 | 1/3 | 1 | 1 |

## 5. What independent allocation permits

If an independent pilot chooses $R$ and $N$ before fresh main draws, conditioning on the pilot preserves the target mean. The unconditional variance averages $A/R+B/(RN)$ over the chosen counts. Inserting average counts into that nonlinear expression generally changes the answer.

The first calculation enumerates an independent two-outcome pilot. The second calculation shows why stopping on realised execution costs needs a different argument. A fair observation $X\in\{-1,1\}$ has cost $(3+X)/2$, and sampling stops when accumulated cost reaches two. The stopped sum has expectation zero, but its average has expectation $1/4$. Optional sampling of the sum therefore does not establish unbiasedness after division by a random sample size. This constructed example identifies a possible mechanism; it does not establish that the measured processor timings in the research were coupled to financial outcomes.

In [5]:
pilot_law = probability.FiniteLaw((((1, 1), Q(1, 3)), ((2, 3), Q(2, 3))))
pilot_selected = probability.pilot_selected_mean(training_law, conditional_gains, pilot_law)
expected_variance = (
    components.a*pilot_law.expectation(lambda counts: Q(1, counts[0]))
    + components.beta*pilot_law.expectation(lambda counts: Q(1, counts[0]*counts[1])))
assert pilot_selected.expectation() == components.mean
assert pilot_selected.variance() == expected_variance
stopped = probability.cost_stopped_signs()
stopped_sum = stopped.expectation(lambda outcome: outcome[1])
stopped_mean = stopped.expectation(lambda outcome: outcome[1]/outcome[0])
assert stopped_sum == 0 and stopped_mean == Q(1, 4)
show_table(['Exact finite-law quantity', 'Value'], [
    ('Independent-pilot estimator mean', str(pilot_selected.expectation())),
    ('Independent-pilot estimator variance', str(pilot_selected.variance())),
    ('Cost-stopped sum expectation', str(stopped_sum)),
    ('Cost-stopped average expectation', str(stopped_mean))])

| Exact finite-law quantity | Value |
| --- | --- |
| Independent-pilot estimator mean | 1/3 |
| Independent-pilot estimator variance | 56/27 |
| Cost-stopped sum expectation | 0 |
| Cost-stopped average expectation | 1/4 |

## 6. Allocation under a stipulated cost function

For positive $A,B,c_T,c_E$, suppose the available budget after setup is $C-c_0>0$ and a complete training pair with $N$ evaluation paths costs $c_T+c_EN$. Substituting $R=(C-c_0)/(c_T+c_EN)$ gives

$$V(N)=\frac{Ac_T+Bc_E+Ac_EN+Bc_T/N}{C-c_0}.$$

Differentiation gives $V'(N)=\{Ac_E-Bc_T/N^2\}/(C-c_0)$ and a positive second derivative. Thus the interior continuous optimum is $N^*=\sqrt{Bc_T/(Ac_E)}$. I use this elementary calculation to interpret the trade-off, without claiming theoretical novelty. Integer counts and measured non-affine costs require a discrete comparison.

The example below has $A=B=4$, setup cost one, pair cost $9+N$ and total budget 73. These are abstract cost units, not processor timings. I compare the reusable allocation selector with enumeration of every feasible pair of integer counts.

In [6]:
from qf_research.optimal_stopping.training_allocation import choose_allocation, continuous_optimum

path_counts = (1, 2, 3, 6, 9)
abstract_costs = {n: 9+n for n in path_counts}
continuous = continuous_optimum(4., 4., 9., 1., 73., 1.)
allocation = choose_allocation(4., 4., 1., abstract_costs,
    reference_repetitions=4, reference_paths=9, min_repetitions=1, max_repetitions=12)
oracle = min((Q(4, r)+Q(4, r*n), n, r)
    for n in path_counts for r in range(1, 13) if 1+r*(9+n) <= 73)
chosen = allocation['selected']
assert (chosen['evaluation_paths'], chosen['repetitions']) == oracle[1:] == (3, 6)
assert np.isclose(continuous['minimum_variance'], float(oracle[0]), rtol=1e-14)
show_table(['Paths N', 'Pairs R', 'Abstract total cost', 'Predicted variance'], [
    (row['evaluation_paths'], row['repetitions'], f"{row['total_cpu_seconds']:.0f}",
     f"{row['predicted_variance']:.6f}") for row in allocation['candidates']])

| Paths N | Pairs R | Abstract total cost | Predicted variance |
| --- | --- | --- | --- |
| 1 | 7 | 71 | 1.142857 |
| 2 | 6 | 67 | 1.000000 |
| 3 | 6 | 73 | 0.888889 |
| 6 | 4 | 61 | 1.166667 |
| 9 | 4 | 73 | 1.111111 |

## 7. The released allocation and CPU findings

The allocation comparison used fixed counts and new simulations conditional on three previously inspected financial specifications. It supplied no additional market observations. The next cell checks its released case coverage, study counts and selected arithmetic identities at the displayed precision. These financial aggregates cannot reproduce the original policy training without the licensed archive.

Source for the historical financial aggregates in this section: LSEG Workspace, accessed through the University of St Andrews subscription; my calculations. The financial specifications use the 10 September 2026 observation window. The earlier stipulated examples are independently chosen mathematical inputs. The CPU ledger contains original computational measurements rather than provider quotes. Repeating an aggregate table in an offline notebook does not turn its source-dependent contents into an unrestricted dataset. The [publication policy](../docs/publication_policy.md) applies to the notebook and its saved outputs as a combined release.

For CPU prediction, I reconstruct the results from the [eighteen-workload ledger](../paper/cpu_timings.json), using exact rational representations of the retained binary timer measurements. Each actual cost $C_i$ includes validation setup and the recorded training and evaluation work. A local forecast adds measured calibration setup to the registered number of training pairs multiplied by the mean cost of sixteen calibration pairs. The historical forecasts are retained scalar predictions; their original pilot is not reconstructed here. Mean absolute relative error is $\frac{100}{18}\sum_{i=1}^{18}\frac{|C_i-\widehat C_i|}{\widehat C_i}$, with forecast cost in the denominator. The [timing supplement](../paper/cpu_timings.md) gives the complete definitions and accounting boundaries.

The first reconstructed table reproduces Table 9 of the paper. The remaining tables show declarations of affordability, actual expenditure and the workload exceeding its original budget. An upper allowance of $1.1\widehat C_i$ can contain the observed cost while being too conservative to admit a workload. Smaller average error also need not ensure affordability. Calibration is charged separately from the forecast's validation target. Warm-up and process overhead remain global costs and are not apportioned to workload budgets. The single-calibration comparisons are accounting comparisons within the recorded experiment, not separate executions.

In [7]:
released = json.loads((PROJECT/'paper/numerical_results.json').read_text())
summary_checks.check(released)
rendered = summary_checks.render(released)
display(Markdown(rendered['prospective_allocation']))

cpu_checks = module_from_file('notebook_cpu_timings',
    PROJECT/'paper/verification/cpu_timings.py')
cpu = cpu_checks.reconstruct(cpu_checks.read(PROJECT/'paper/cpu_timings.json'))
cpu_checks.check_summary(cpu, released)
cpu_tables = cpu_checks.render(cpu)
cpu_checks.check_markdown((PROJECT/'paper/cpu_timings.md').read_text(), cpu_tables)
for name in ('prediction', 'affordability', 'charged'):
    display(Markdown(cpu_tables[name]))
show_table(['Reconstructed CPU-study quantity', 'Count'], [
    ('Validation workloads', cpu['counts']['validation_arms']),
    ('Validation training pairs', cpu['counts']['validation_pairs']),
    ('Calibration training pairs', cpu['counts']['calibration_pairs']),
    ('Warm-up training pairs', cpu['counts']['warmup_pairs'])])
display(Markdown(cpu_tables['costs']))

overruns = [row for row in cpu['workloads'] if row['budget_excess_cpu'] > 0]
assert len(overruns) == 1
assert len(overruns) + cpu['affordability']['validation'] == cpu['counts']['validation_arms']
show_table(['Pass', 'Position', 'Case', 'Evaluation paths', 'Training pairs',
            'Actual CPU seconds', 'Budget CPU seconds', 'Excess CPU seconds'], [
    (row['pass_id']+1, row['position']+1, row['case'], row['evaluation_paths'],
     row['training_pairs'], cpu_checks.number(row['actual_cpu'], 3),
     cpu_checks.number(row['budget_cpu'], 3),
     cpu_checks.number(row['budget_excess_cpu'], 3)) for row in overruns])

| Case | Pairs, 128 paths | Pairs, 256 paths | Pairs, 4,096 paths | Variance ratio, 128 | Variance ratio, 256 | CPU ratio, 256 | Variance × CPU ratio, 256 |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 5 | 776 | 687 | 91 | 0.169888 | 0.167421 | 1.510736 | 0.252928 |
| 6 | 674 | 632 | 141 | 0.396693 | 0.340606 | 0.901768 | 0.307147 |
| 10 | 438 | 399 | 140 | 0.364378 | 0.330904 | 0.655844 | 0.217021 |

| Forecast | Within ten per cent | Within upper allowance | Mean absolute relative error (per cent) | Actual / forecast range |
| --- | --- | --- | --- | --- |
| Historical | 1/18 | 18/18 | 29.065 | [0.5154, 1.0782] |
| Pass-start calibration | 16/18 | 16/18 | 3.669 | [0.9237, 1.2081] |
| Immediate calibration | 17/18 | 17/18 | 2.841 | [0.9576, 1.2261] |

| Forecast | Point forecast affordable | False point declarations | Upper allowance affordable | False allowance declarations |
| --- | --- | --- | --- | --- |
| Historical | 18/18 | 1/18 | 0/18 | 0/18 |
| Pass-start calibration | 18/18 | 1/18 | 18/18 | 1/18 |
| Immediate calibration | 18/18 | 1/18 | 17/18 | 1/18 |

| Charge compared with the unchanged workload budget | Affordable workloads |
| --- | --- |
| Validation only | 17/18 |
| Validation and pass-start calibration | 17/18 |
| Validation and immediate calibration | 17/18 |
| Validation and both calibration schemes | 14/18 |

| Reconstructed CPU-study quantity | Count |
| --- | --- |
| Validation workloads | 18 |
| Validation training pairs | 7956 |
| Calibration training pairs | 576 |
| Warm-up training pairs | 4 |

| Component | Seconds |
| --- | --- |
| Validation numerical CPU | 700.274 |
| Pass-start calibration CPU | 44.623 |
| Immediate calibration CPU | 44.323 |
| Warm-up CPU | 0.332 |
| Total numerical CPU | 789.552 |
| Other measured process CPU | 22.106 |
| Whole-process CPU | 811.658 |
| Whole-process elapsed time | 842.795 |

| Pass | Position | Case | Evaluation paths | Training pairs | Actual CPU seconds | Budget CPU seconds | Excess CPU seconds |
| --- | --- | --- | --- | --- | --- | --- | --- |
| 1 | 7 | 10 | 4096 | 140 | 54.487 | 50.636 | 3.852 |

## 8. Interpretation

The funded-model run demonstrates an executable route from specified financial inputs to fitted policies, martingale loss assessment and repeated-training uncertainty. The exact examples explain which independence and sampling assumptions that route needs. The released financial summaries address a larger conditional experiment; they cannot be inferred from this smaller mathematical demonstration.

The allocation comparison supports a lower estimated variance times CPU product for the studied procedures and specifications. It does not establish a universal allocation optimum, equal realised budgets or verified finite-sample coverage. The CPU reconstruction shows that recent calibration improved average prediction in the recorded session, while preserving one budget failure. Charging both recorded calibration schemes reduces affordability from seventeen to fourteen of the eighteen workloads under their unchanged budgets. These counts exclude global warm-up and process overhead. The experiment supplies neither a deterministic deadline guarantee nor evidence of a causal processor-order effect.

The timing ledger permits independent reconstruction of the published arithmetic. It does not authenticate the chronology of withheld records or establish the accuracy of the original processor timers. Exact rational arithmetic prevents further rounding in these comparisons; it does not make a measured duration exact in physical time. No original financial or timing experiment is repeated in this section.

Independent validation against a fresh, adequately documented market sample was not completed. The incomplete historical funding construction was excluded. The conditional mathematical and computational conclusions remain assessable through their stated assumptions, code, released summaries and computational measurements, without treating missing constituents as observed inputs.

## 9. Sources and attribution

Least-squares exercise-policy fitting follows Longstaff, F. A. and Schwartz, E. S. (2001), ‘Valuing American options by simulation: a simple least-squares approach’, *Review of Financial Studies*, 14(1), 113–147, [doi:10.1093/rfs/14.1.113](https://doi.org/10.1093/rfs/14.1.113). The martingale assessment follows Haugh, M. B. and Kogan, L. (2004), ‘Pricing American options: a duality approach’, *Operations Research*, 52(2), 258–270, [doi:10.1287/opre.1030.0070](https://doi.org/10.1287/opre.1030.0070).

The [source register](../SOURCES.md) and [contribution review](../paper/contribution_review.md) identify the consulted material and distinguish established methods from the project-specific analysis. The finite-law verifiers, stipulated configuration and numerical kernels are repository implementations. Their checks establish the stated instances and numerical tolerances, without claiming a new general theorem. The [CPU supplement](../paper/cpu_timings.md) attributes the retained experiment and identifies the public arithmetic verifier.